In [ ]:
#165 - Melhorando o Output com Output Parser
## Abaixo o texto com a descrição do produto
feedback_produto = """
Estou muito satisfeito com o Smartphone XYZ Pro. O desempenho é excelente, e o sistema
operacional é rápido e intuitivo. A câmera é um dos principais destaques, especialmente o
modo noturno, que captura imagens incríveis mesmo em baixa iluminação. A duração da bateria
também impressiona, durando facilmente um dia inteiro com uso intenso.
Por outro lado, sinto que o produto poderia ser melhorado em alguns aspectos. A tela,
embora tenha cores vibrantes, parece refletir bastante luz, dificultando o uso sob o sol.
Além disso, o carregador incluído na caixa não oferece carregamento rápido, o que é um ponto
negativo considerando o preço do aparelho
"""

In [ ]:
## Abaixo um template com as definições do que quero que seja extraido da descriçã do produto

from langchain.prompts import ChatPromptTemplate

review_template = ChatPromptTemplate.from_template("""
Para o texto a seguir, extraia as seguintes informações:
produto: Nome do produto mencionado no texto.
características_positivas: Liste todas as características positivas mencionadas sobre o produto.
características_negativas: Liste todas as características negativas mencionadas sobre o produto.
recomendação: O cliente recomenda o produto? Responda True para sim ou False para não.

Texto: {review}

Retorne a resposta no formato JSON
""")

In [ ]:
from langchain_openai.chat_models import ChatOpenAI

In [ ]:
## Abaixo temos o chat sendo executado, e o no parametro review é enviado o feedback_produto que corres
##pode a variavel com o texto a ser extraido(avaliado)
chat = ChatOpenAI()
resposta = chat.invoke(review_template.format_messages(review=feedback_produto))

In [ ]:
## Aqui exibira a saida/reposta com o conteudo do LLM.
## Até esse ponto somente definimos um template para avaliação e extração de um texto
resposta.content

In [ ]:
from langchain.output_parsers import ResponseSchemafrom
from langchain.output_parsers import StructuredOutputParser

In [ ]:
schema_produto = ResponseSchema(
    name="produto",
    type="string",
    description="Nome do produto mencionado no texto"
)

schema_positivas = ResponseSchema(
    name="caracteristicas_positivas",
    type="list",
    description="Liste todas as características positivas mencionadas sobre o produto"
)

schema_negativas = ResponseSchema(
    name="caracteristicas_negativas",
    type="list",
    description="Liste todas as características negativas mencionadas sobre o produto"
)

schema_recomendacao = ResponseSchema(
    name="recomendacao",
    type="bool",
    description="O cliente recomenda o produto? Responda True p/ Sim ou False para Nao"
)

In [ ]:
## aqui eu compilo os 4 esquemas e um unico
response_schema = [schema_produto, schema_positivas, schema_negativas, schema_recomendacao]
output_parser = StructuredOutputParser.from_response_schemas(response_schema)
schema_formatado = output_parser.get_format_instructions()

In [ ]:
print(schema_formatado)

In [ ]:
### Resultado
'''{
  "produto": string // Nome do produto mencionado no texto
  "características_positivas": list // Liste todas as características positivas mencionadas sobre o produto
  "características_negativas": list // Liste todas as características negativas mencionadas sobre o produto
  "recomendação": bool // O cliente recomenda o produto? Responda True para sim ou False para não.
}'''

In [ ]:
review_template2 = ChatPromptTemplate.from_template("""
Para o texto a seguir, extraia as seguintes informações:
produto, caracteristicas_positivas, caracteristicas_negativas e recomendacao

Texto: {review}

{schema}
""", partial_variables={"schema": schema_formatado})


In [ ]:
##Aqui teremos o retorno do LLM considerando o texto no inicio do projeto, sendo considerada
## as caracteristicas do template2 + as definições do schema.
resposta = chat.invoke(review_template2.format_messages(review=feedback_produto))
resposta.content

In [ ]:
resposta_json = output_parser.parse(resposta.content)

In [ ]:
##Aqui conseguimos fazer o parse por tipo de caracteristica(produto, caracteristica positiva, negativa e etc)
resposta_json["produto"]

In [ ]:
resposta_json["caracteristicas_positivas"]

In [ ]:
resposta_json["características_negativas"]